# Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the Leeds IMD data.

In [1]:
import pandas as pd
import numpy as np
import os

OUTPUT_FOLDER = 'output'
INPUT_FILE = os.path.join(OUTPUT_FOLDER, 'Leeds_IMD_LSOA.csv')

print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Data file not found: {INPUT_FILE}. Please run 01_Data_Preparation.ipynb first.")

print(f"\nLoading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns")

EXPLORATORY DATA ANALYSIS

Loading data from output\Leeds_IMD_LSOA.csv...
Loaded 482 rows and 57 columns


## Summary Statistics

In [2]:
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) == 0:
    print("No numeric columns found")
else:
    summary = df[numeric_cols].describe()
    summary.loc['median'] = df[numeric_cols].median()
    summary.loc['std'] = df[numeric_cols].std()
    summary.loc['skewness'] = df[numeric_cols].skew()
    summary


SUMMARY STATISTICS


## IMD Decile Distribution

In [3]:
print("\n" + "=" * 60)
print("IMD DECILE DISTRIBUTION")
print("=" * 60)

# Find decile column
decile_col = None
possible_names = ['IMD Decile', 'Index of Multiple Deprivation Decile', 'Decile']

for col in df.columns:
    if any(name.lower() in col.lower() for name in possible_names):
        decile_col = col
        break

if decile_col is None:
    print("Warning: IMD Decile column not found")
else:
    decile_counts = df[decile_col].value_counts().sort_index()
    print(f"\nNumber of LSOAs in each IMD Decile:")
    print(decile_counts)
    print(f"\nTotal LSOAs: {decile_counts.sum()}")


IMD DECILE DISTRIBUTION

Number of LSOAs in each IMD Decile:
Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
1     114
2      48
3      42
4      21
5      45
6      40
7      56
8      42
9      41
10     33
Name: count, dtype: int64

Total LSOAs: 482


## Deprivation Domain Analysis

In [4]:
print("\n" + "=" * 60)
print("DEPRIVATION DOMAIN ANALYSIS")
print("=" * 60)

# Find domain score columns
domain_names = ['Income', 'Employment', 'Education', 'Health', 'Crime', 'Housing', 'Environment']
domain_scores = {}

for domain in domain_names:
    matching_cols = [col for col in df.columns 
                    if domain.lower() in col.lower() and 'score' in col.lower()]
    if matching_cols:
        domain_scores[domain] = matching_cols[0]

if not domain_scores:
    print("Warning: Could not find domain score columns")
else:
    # Calculate statistics for each domain
    domain_analysis = []
    
    for domain_name, col_name in domain_scores.items():
        if col_name in df.columns:
            domain_data = df[col_name].dropna()
            
            domain_analysis.append({
                'Domain': domain_name,
                'Mean Score': domain_data.mean(),
                'Median Score': domain_data.median(),
                'Std Dev': domain_data.std(),
                'Min Score': domain_data.min(),
                'Max Score': domain_data.max(),
                'LSOAs with Data': len(domain_data)
            })
    
    domain_df = pd.DataFrame(domain_analysis)
    domain_df = domain_df.sort_values('Mean Score', ascending=False)
    
    print("\nDomain Statistics (sorted by mean score - higher = more deprived):")
    print(domain_df.to_string(index=False))


DEPRIVATION DOMAIN ANALYSIS

Domain Statistics (sorted by mean score - higher = more deprived):
     Domain  Mean Score  Median Score   Std Dev  Min Score  Max Score  LSOAs with Data
Environment   34.161012       32.4555 17.044904      3.861     81.847              482
  Education   26.245494       18.5395 23.619542      0.321     89.352              482
    Housing   15.231541       15.2185  6.459307      1.340     42.332              482
      Crime    0.657726        0.6000  0.845120     -2.455      2.755              482
     Health    0.408365        0.3385  0.708222     -1.496      3.215              482
     Income    0.144060        0.0990  0.110654      0.009      0.459              482
 Employment    0.111102        0.0810  0.075725      0.007      0.343              482


## Population Characteristics

In [5]:
print("\n" + "=" * 60)
print("POPULATION CHARACTERISTICS")
print("=" * 60)

# Find population-related columns
pop_cols = {}

for pop_type in ['Children', 'Working', 'Older', 'Population']:
    matching_cols = [col for col in df.columns 
                    if pop_type.lower() in col.lower() and 
                    ('count' in col.lower() or 'number' in col.lower() or 'pop' in col.lower())]
    if matching_cols:
        pop_cols[pop_type] = matching_cols[0]

if not pop_cols:
    print("Warning: Could not find population columns")
else:
    print("\nPopulation Statistics:")
    for pop_type, col_name in pop_cols.items():
        if col_name in df.columns:
            pop_data = df[col_name].dropna()
            total = pop_data.sum()
            mean = pop_data.mean()
            median = pop_data.median()
            
            print(f"\n{pop_type} ({col_name}):")
            print(f"  Total: {total:,.0f}")
            print(f"  Mean per LSOA: {mean:,.0f}")
            print(f"  Median per LSOA: {median:,.0f}")


POPULATION CHARACTERISTICS

Population Statistics:

Working (Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners) ):
  Total: 473,004
  Mean per LSOA: 981
  Median per LSOA: 880

Older (Older population aged 60 and over: mid 2015 (excluding prisoners)):
  Total: 155,158
  Mean per LSOA: 322
  Median per LSOA: 320

Population (Total population: mid 2015 (excluding prisoners)):
  Total: 771,857
  Mean per LSOA: 1,601
  Median per LSOA: 1,524


## Save Summary Statistics

In [6]:
output_file = os.path.join(OUTPUT_FOLDER, 'summary_statistics.txt')

with open(output_file, 'w') as f:
    f.write("LEEDS HEALTH INEQUALITIES PROJECT - SUMMARY STATISTICS\n")
    f.write("=" * 60 + "\n\n")
    
    if 'summary' in locals():
        f.write("SUMMARY STATISTICS\n")
        f.write("-" * 60 + "\n")
        f.write(summary.to_string())
        f.write("\n\n")
    
    if 'decile_counts' in locals():
        f.write("IMD DECILE DISTRIBUTION\n")
        f.write("-" * 60 + "\n")
        f.write(decile_counts.to_string())
        f.write("\n\n")
    
    if 'domain_df' in locals():
        f.write("DEPRIVATION DOMAIN ANALYSIS\n")
        f.write("-" * 60 + "\n")
        f.write(domain_df.to_string(index=False))
        f.write("\n\n")

print(f"\nSummary statistics saved to {output_file}")

print("\n" + "=" * 60)
print("EDA COMPLETE")
print("=" * 60)


Summary statistics saved to output\summary_statistics.txt

EDA COMPLETE
